In [ ]:
from crewai.flow.flow import Flow,start
class SimpleFlow(Flow):
   
   @start()
   def initialize(self):
      print("FLow started")

flow = SimpleFlow()
await flow.kickoff_async()

In [ ]:
from crewai.flow.flow import Flow, listen, start

class SequentialFlow(Flow):

    @start()
    def first_task(self):
        print("Step1: Fetching data")
        return "data_fetched"

    @listen(first_task)
    def second_task(self, result):
        print(f"Step2: Processing {result}")

flow = SequentialFlow()
await flow.kickoff_async()



HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30)


In [ ]:
from crewai.flow.flow import Flow, start, listen, or_

class OrFlow(Flow):

    @start()
    def fetch_from_api(self):
        return "API data"

    @start()
    def read_from_db(self):
        return "Databse record"
    
    @listen(or_(fetch_from_api, read_from_db))
    def process_data(self, result):
        print(f"Processing: {result}")

flow= OrFlow()
await flow.kickoff_async()


In [ ]:
# this code demonstrates the use of and_ variable
from crewai.flow.flow import Flow, start, listen, and_

class AndFlow(Flow):

    @start()
    def step_one(self):
        print(f"Step 1: Collecting user input")
        return "User data"

    @start()
    def step_second(self):
        print(f"Step 2: Validating the input")
        return "Validated data"  

    @listen(and_(step_one, step_second))
    def final_step(self):
        print("All conditions are met. Processing with the final step")


flow= AndFlow()
await flow.kickoff_async()


In [ ]:
#router is used to direct the issue based on the urgency of the problem
import random
from crewai.flow.flow import start, listen, Flow, router

class RouterFlow(Flow):
    
    @start()
    def classify_request(self):
        request_type = random.choice(["urgent", "normal"])
        print(f"Request classified as:{request_type}")
        return request_type

    @router(classify_request)
    def handle_request(self, request_type):
        if request_type == "urgent":
            return "handle_urgent"
        else:
            return "handle_normal"
        
    @listen("handle_urgent")
    def urgent_handler(self):
        print("Handling urgent request")

    @listen("handle_normal")
    def normal_handler(self):
        print("Handling normal request")

flow = RouterFlow()
await flow.kickoff_async()



In [ ]:
#unstructured state state dictionary (self.state); which persist across the steps.
from crewai.flow.flow import Flow, listen, start

class StateFlow(Flow):

    @start()
    def initialize_state(self):
        self.state['count'] = 1
        print(f"Initial count: {self.state['count']}")

    @listen(initialize_state)
    def increment_count(self):
        self.state["count"] += 1
        print(f"Count incremented: {self.state['count']}")

flow = StateFlow()
await flow.kickoff_async()

In [ ]:
#structure state example

from crewai.flow.flow import Flow, listen, start
from pydantic import BaseModel

# Defining a structured state model
class CounterState(BaseModel):
    count: int=0

class StructuredStateFlow(Flow[CounterState]):

    @start()
    def initialize_state(self):
        print(f"Initial count: {self.state.count}")
        self.state.count = 1

    @listen(initialize_state)
    def incerement_state(self):
        print(f"Final Count: {self.state.count}")
        self.state.count +=1

flow = StructuredStateFlow()
await flow.kickoff_async()